# Importando as bibliotecas

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import optuna
import mlflow

import warnings
warnings.filterwarnings("ignore")

c:\Users\felip\miniconda3\envs\mlops\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregando a base de dados

In [2]:
data = pd.read_csv('data/bike.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  cnt         17379 non-null  int64  
dtypes: float64(4), int64(10), object(1)
memory usage: 2.0+ MB


In [3]:
data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,1/1/2011,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16
1,2,1/1/2011,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40
2,3,1/1/2011,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32
3,4,1/1/2011,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13
4,5,1/1/2011,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


# Preparando o Dataset - Treino e Teste

In [4]:
# Converte a coluna 'dteday' para o tipo datetime
data['dteday'] = pd.to_datetime(data['dteday'])

In [5]:
# Divisão 80% treino e 20% teste
train = data.loc[data['dteday']< '2012-08-10']
test = data.loc[data['dteday']>= '2012-08-10']

In [6]:
# Porcentagem dos dados de treino
round(train.shape[0] / data.shape[0], 2)

0.8

In [7]:
train.drop(['instant', 'dteday'], axis=1, inplace=True)
test.drop(['instant', 'dteday'], axis=1, inplace=True)

# Validação Cruzada p/ teste de Modelos

In [8]:
# Separando as features e o target de treino
features_train = train.drop('cnt', axis=1)
target_train = train['cnt']

# Separando as features e o target de teste
features_test = test.drop('cnt', axis=1)
target_test = test['cnt']

In [9]:
def cross_validate_rmse(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=cv)
    rmse_scores = -scores
    mean_rmse = rmse_scores.mean()
    std_rmse = rmse_scores.std()
    return rmse_scores, mean_rmse, std_rmse

# Decision Tree
rmse_scores, mean_rmse, std_rmse = cross_validate_rmse(DecisionTreeRegressor(random_state=42), features_train, target_train)
print(f'Decision Tree: RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')

# Random Forest
rmse_scores, mean_rmse, std_rmse = cross_validate_rmse(RandomForestRegressor(n_estimators=1000, random_state=42, n_jobs=-1), features_train, target_train)
print(f'Random Forest: RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')

# XGBoost with DataFrame
rmse_scores, mean_rmse, std_rmse = cross_validate_rmse(xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', seed=42), features_train, target_train)
print(f'XGBoost (DataFrame): RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')

# XGBoost with Dmatrix
dtrain = xgb.DMatrix(features_train, label=target_train)
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
}
cv_results = xgb.cv(params, dtrain, num_boost_round=1000, nfold=5, early_stopping_rounds=10)
mean_rmse = cv_results['test-rmse-mean'].min()
std_rmse = cv_results['test-rmse-std'].min()
print(f'XGBoost (DMatrix): RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')


Decision Tree: RMSE médio: 90.62, Desvio padrão: 15.20
Random Forest: RMSE médio: 71.25, Desvio padrão: 14.70
XGBoost (DataFrame): RMSE médio: 62.06, Desvio padrão: 7.56
XGBoost (DMatrix): RMSE médio: 36.87, Desvio padrão: 1.39


# Experiment Tracking com MLflow e XGBoost

In [10]:
# Instancia o MLflow
mlflow.set_experiment("bike_rental")
mlflow.set_tracking_uri("http://localhost:5000")

2025/02/20 16:57:27 INFO mlflow.tracking.fluent: Experiment with name 'bike_rental' does not exist. Creating a new experiment.


In [20]:
# Otimização de hiperparâmetros com Optuna e registro no MLflow

def objective(trial):
    # Define os hiperparâmetros a serem otimizados
    param = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 42,
        'max_depth': trial.suggest_int("max_depth", 3, 10),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.3),
        'gamma': trial.suggest_float("gamma", 0.0, 5.0),
        'min_child_weight': trial.suggest_int("min_child_weight", 1, 10),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'colsample_bytree': trial.suggest_float("colsample_bytree", 0.5, 1.0)
    }
    
    # Inicia um run MLflow aninhado para registrar a trial corrente
    with mlflow.start_run(nested=True):

        # Registra os hiperparâmetros no MLflow
        mlflow.log_params(param)
        
        # Executa a validação cruzada com xgb.cv usando dtrain (definido em célula anterior)
        cv_results = xgb.cv(
            params=param,
            dtrain=dtrain,
            num_boost_round=1000,
            nfold=5,
            early_stopping_rounds=10,
            verbose_eval=False
        )
        
        # Retorna o menor RMSE obtido na validação cruzada
        best_rmse = cv_results['test-rmse-mean'].min()
        mlflow.log_metric("best_rmse", best_rmse)
    
    return best_rmse

# Cria um estudo Optuna e otimiza a função objetivo
study_dmatrix = optuna.create_study(direction="minimize")
study_dmatrix.optimize(objective, n_trials=50)

print("Melhor resultado:")
print(study_dmatrix.best_trial)

[I 2025-02-20 17:12:22,418] A new study created in memory with name: no-name-2c5d48ad-083e-48f0-a7e8-71543af10c86
[I 2025-02-20 17:12:24,301] Trial 0 finished with value: 40.44427104517274 and parameters: {'max_depth': 9, 'learning_rate': 0.2984116907046961, 'gamma': 1.9987419929143484, 'min_child_weight': 2, 'subsample': 0.7876169835103024, 'colsample_bytree': 0.6314287014771984}. Best is trial 0 with value: 40.44427104517274.


🏃 View run unruly-sponge-266 at: http://localhost:5000/#/experiments/788856940458785600/runs/26e68c5748a743b7baafd4f129008839
🧪 View experiment at: http://localhost:5000/#/experiments/788856940458785600


[I 2025-02-20 17:12:28,182] Trial 1 finished with value: 35.40673301808342 and parameters: {'max_depth': 6, 'learning_rate': 0.09479337929984202, 'gamma': 3.148563880230442, 'min_child_weight': 5, 'subsample': 0.8997890036581935, 'colsample_bytree': 0.8955866274115096}. Best is trial 1 with value: 35.40673301808342.


🏃 View run nebulous-crane-519 at: http://localhost:5000/#/experiments/788856940458785600/runs/f446741f95cc4a4788ec9b3ac1b7781e
🧪 View experiment at: http://localhost:5000/#/experiments/788856940458785600


[I 2025-02-20 17:12:34,682] Trial 2 finished with value: 35.606441214204224 and parameters: {'max_depth': 8, 'learning_rate': 0.05494035356415756, 'gamma': 1.2404745667271233, 'min_child_weight': 2, 'subsample': 0.9409221310587981, 'colsample_bytree': 0.5435838728760454}. Best is trial 1 with value: 35.40673301808342.


...

🏃 View run classy-shrew-568 at: http://localhost:5000/#/experiments/788856940458785600/runs/b427c56142c34c259d26ffaf4aece936
🧪 View experiment at: http://localhost:5000/#/experiments/788856940458785600


[I 2025-02-20 17:17:46,698] Trial 48 finished with value: 34.75276407145414 and parameters: {'max_depth': 8, 'learning_rate': 0.05852711857782946, 'gamma': 3.598580493675275, 'min_child_weight': 6, 'subsample': 0.7851612800120322, 'colsample_bytree': 0.9294970638251828}. Best is trial 21 with value: 34.40783537527264.


🏃 View run languid-carp-756 at: http://localhost:5000/#/experiments/788856940458785600/runs/22c440728eca40138cc82dd0ea7c8e4b
🧪 View experiment at: http://localhost:5000/#/experiments/788856940458785600


[I 2025-02-20 17:17:48,052] Trial 49 finished with value: 38.06276292945036 and parameters: {'max_depth': 10, 'learning_rate': 0.2636507014431756, 'gamma': 3.9976942093289645, 'min_child_weight': 9, 'subsample': 0.5933007097658699, 'colsample_bytree': 0.9971035028200741}. Best is trial 21 with value: 34.40783537527264.


🏃 View run casual-bass-858 at: http://localhost:5000/#/experiments/788856940458785600/runs/e1de099bef1443788b1e1d4b72196cf4
🧪 View experiment at: http://localhost:5000/#/experiments/788856940458785600
Melhor resultado:
FrozenTrial(number=21, state=TrialState.COMPLETE, values=[34.40783537527264], datetime_start=datetime.datetime(2025, 2, 20, 17, 14, 24, 902089), datetime_complete=datetime.datetime(2025, 2, 20, 17, 14, 32, 232698), params={'max_depth': 9, 'learning_rate': 0.04032633610146599, 'gamma': 3.103460583323776, 'min_child_weight': 9, 'subsample': 0.732925837962594, 'colsample_bytree': 0.8765889068068179}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'gamma': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'subsample': FloatDistrib

In [12]:
# Exibe os 10 melhores resultados
study_dmatrix.trials_dataframe().sort_values(by="value", ascending=True).head(10)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
36,36,34.279833,2025-02-20 17:01:28.647891,2025-02-20 17:01:41.466488,0 days 00:00:12.818597,0.862370,4.962442,0.020595,10,9,0.710493,COMPLETE
33,33,34.352235,2025-02-20 17:01:00.300384,2025-02-20 17:01:13.052017,0 days 00:00:12.751633,0.843314,4.959555,0.018792,10,9,0.576328,COMPLETE
37,37,34.356974,2025-02-20 17:01:41.466488,2025-02-20 17:01:55.601557,0 days 00:00:14.135069,0.924169,4.378917,0.017317,10,9,0.712684,COMPLETE
38,38,34.374015,2025-02-20 17:01:55.601557,2025-02-20 17:02:10.848340,0 days 00:00:15.246783,0.915304,4.397476,0.014964,10,10,0.705173,COMPLETE
35,35,34.385417,2025-02-20 17:01:17.667115,2025-02-20 17:01:28.647891,0 days 00:00:10.980776,0.915892,4.380986,0.024204,10,9,0.699881,COMPLETE
42,42,34.388339,2025-02-20 17:02:46.380636,2025-02-20 17:02:58.334803,0 days 00:00:11.954167,0.869016,4.964180,0.021445,10,10,0.774608,COMPLETE
39,39,34.402127,2025-02-20 17:02:10.848340,2025-02-20 17:02:24.448344,0 days 00:00:13.600004,0.984465,4.283346,0.015669,9,10,0.727173,COMPLETE
32,32,34.443896,2025-02-20 17:00:51.613147,2025-02-20 17:01:00.300384,0 days 00:00:08.687237,0.849069,4.929645,0.029467,10,8,0.615685,COMPLETE
47,47,34.466450,2025-02-20 17:03:11.618053,2025-02-20 17:03:25.569713,0 days 00:00:13.951660,0.963294,4.344984,0.021490,10,9,0.851884,COMPLETE
43,43,34.534531,2025-02-20 17:02:58.334803,2025-02-20 17:03:04.648252,0 days 00:00:06.313449,0.927927,4.422444,0.048903,9,9,0.709702,COMPLETE


In [13]:
study_dmatrix.best_params

{'max_depth': 10,
 'learning_rate': 0.02059515711564996,
 'gamma': 4.96244196764165,
 'min_child_weight': 9,
 'subsample': 0.7104927008270152,
 'colsample_bytree': 0.8623696933619331}